# 音声入力から感情・VAD出力まで
このNotebookが利用者向けの唯一の実行入口です。1回につき1データセットを扱います。`DEMO_MODE=True` の結果は動作確認専用で、研究結果には使用できません。

## 1. 設定

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from vad_downstream.notebook_pipeline import *
DEMO_MODE = True  # 実データではFalseに変更
COLUMNS = ColumnConfig(audio='audio', speaker='speaker', emotion='emotion', valence='valence', arousal='arousal', dominance=None)
CONFIG = NotebookConfig(demo_mode=DEMO_MODE, columns=COLUMNS, epochs=2, device='cpu')
print(CONFIG)

## 2. デモデータ準備（合成音声・仮特徴抽出器）実データ利用時はこのセルは何も変更しません。デモ生成物は `runs/notebooks/` 以下にのみ保存されます。

In [ ]:
import numpy as np, pandas as pd, soundfile as sf
run_dir = ROOT / CONFIG.output_dir
run_dir.mkdir(parents=True, exist_ok=True)
if DEMO_MODE:
    demo_audio, demo_infer = run_dir/'demo_audio', run_dir/'demo_inference'
    demo_audio.mkdir(exist_ok=True); demo_infer.mkdir(exist_ok=True)
    rows=[]
    for speaker in range(6):
        for emotion_id, emotion in enumerate(['happy', 'sad']):
            name=f's{speaker}_{emotion}.wav'; t=np.arange(3200)/16000
            wav=(0.1*np.sin(2*np.pi*(180+speaker*20+emotion_id*100)*t)).astype('float32')
            sf.write(demo_audio/name, wav, 16000)
            rows.append({'audio':name,'speaker':f's{speaker}','emotion':emotion,'valence':speaker+emotion_id,'arousal':10-speaker+emotion_id})
    sf.write(demo_infer/'sample_a.wav', np.zeros(3200,dtype='float32'), 16000)
    sf.write(demo_infer/'sample_b.wav', 0.05*np.sin(2*np.pi*300*np.arange(3200)/16000).astype('float32'), 16000)
    pd.DataFrame(rows).to_csv(run_dir/'demo_annotations.csv', index=False)
    CONFIG.audio_dir=str(demo_audio); CONFIG.annotation_csv=str(run_dir/'demo_annotations.csv'); CONFIG.inference_dir=str(demo_infer)
    print('デモモード: この結果は研究結果ではありません。')

## 3. CSV・音声の一括検証欠損列、欠損/破損音声、不正値、16kHzモノラル以外があれば一覧表示して停止します。感情名と分類数はCSVから自動取得します。

In [ ]:
annotations, emotion_labels = load_and_validate_annotations(ROOT/CONFIG.annotation_csv if not Path(CONFIG.annotation_csv).is_absolute() else CONFIG.annotation_csv, ROOT/CONFIG.audio_dir if not Path(CONFIG.audio_dir).is_absolute() else CONFIG.audio_dir, COLUMNS)
display(annotations.head()); print('感情ラベル:', emotion_labels, '分類数:', len(emotion_labels))

## 4. seed固定・話者単位のtrain/valid/test分割話者リークと、trainに全感情が存在することを検査します。

In [ ]:
split_frame = speaker_split(annotations, COLUMNS.speaker, COLUMNS.emotion, CONFIG.valid_ratio, CONFIG.test_ratio, CONFIG.seed)
display(pd.crosstab(split_frame['split'], split_frame[COLUMNS.emotion])); display(split_frame.groupby('split')[COLUMNS.speaker].unique())

## 5. train splitだけでV/A/DをMin-Max正規化係数は `[-1,1]` 変換と元尺度への逆変換に共用し、checkpointにも保存します。D列がなければDは未学習です。

In [ ]:
column_map={'valence':COLUMNS.valence,'arousal':COLUMNS.arousal,'dominance':COLUMNS.dominance}
normalizer=TrainMinMaxNormalizer.fit(split_frame[split_frame.split=='train'], column_map)
split_frame=normalizer.transform_frame(split_frame, column_map)
display(pd.DataFrame(normalizer.to_dict()));
if not normalizer.trained['dominance']: print(DOMINANCE_WARNING)

## 6. emotion2vec特徴抽出とキャッシュキャッシュキーは音声パス・更新情報・encoder checkpointに対応します。再実行時は抽出済み特徴を再利用します。本物のencoderは固定されます。

In [ ]:
if DEMO_MODE:
    extractor, encoder_id = demo_feature_extractor, 'DEMO_FAKE_ENCODER_NOT_FOR_RESEARCH'
else:
    from vad_downstream.inference import build_audio_encoder
    encoder=build_audio_encoder(CONFIG.encoder_model_dir, CONFIG.encoder_checkpoint, CONFIG.device)
    for parameter in encoder.parameters(): parameter.requires_grad=False
    extractor=make_emotion2vec_extractor(encoder, CONFIG.device); encoder_id=str(Path(CONFIG.encoder_checkpoint).resolve())
cache=FeatureCache(run_dir/'feature_cache', encoder_id, extractor)
for path in split_frame.audio_path: cache.get(path)
print('特徴キャッシュ完了:', run_dir/'feature_cache')

## 7. Dataset・モデル構築と1バッチ確認抽出済み768次元特徴の上に、独立した感情・V・A・Dヘッドだけを構築します。

In [ ]:
datasets={s:make_dataset(split_frame[split_frame.split==s].reset_index(drop=True), emotion_labels, cache, COLUMNS) for s in ['train','valid','test']}
loaders={s:make_loader(ds, CONFIG.batch_size, s=='train', CONFIG.seed) for s,ds in datasets.items()}
model=ParallelEmotionVADClassifier(len(emotion_labels), hidden_dim=CONFIG.hidden_dim).to(CONFIG.device)
first_batch=next(iter(loaders['train'])); first_output=model(first_batch['net_input']['feats'].to(CONFIG.device), first_batch['net_input']['padding_mask'].to(CONFIG.device))
print({k:tuple(v.shape) for k,v in first_output.items()})

## 8. epoch学習D教師なしの場合、Dヘッドはoptimizerから除外され固定されます。

In [ ]:
history=train_epochs(model, loaders['train'], loaders['valid'], CONFIG.epochs, CONFIG.learning_rate, CONFIG.device, emotion_labels)
save_json(history, run_dir/'training_history.json'); display(pd.json_normalize(history))

## 9. 評価・学習曲線・confusion matrixWA、UA、weighted F1、V/A/D CCCを表示します。D教師なしならDominance CCCは空欄です。

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from vad_downstream.parallel_training import evaluate
test_metrics=evaluate(model, loaders['test'], CONFIG.device, class_labels=emotion_labels)
save_json(test_metrics, run_dir/'test_metrics.json'); display(pd.Series(test_metrics).drop('confusion_matrix'))
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].plot([h['epoch'] for h in history],[h['train']['loss'] for h in history],label='train'); axes[0].plot([h['epoch'] for h in history],[h['valid']['loss'] for h in history],label='valid'); axes[0].set_title('Learning curve'); axes[0].legend()
sns.heatmap(test_metrics['confusion_matrix'],annot=True,fmt='d',xticklabels=emotion_labels,yticklabels=emotion_labels,ax=axes[1]); axes[1].set(xlabel='prediction',ylabel='target',title='Confusion matrix')
fig.tight_layout(); fig.savefig(run_dir/'evaluation_graphs.png',dpi=150); plt.show()

## 10. checkpoint・ラベル・正規化情報の保存既存並列checkpointのキーを維持し、Notebook用メタデータを追加します。

In [ ]:
from dataclasses import asdict
from vad_downstream.parallel_training import save_parallel_checkpoint
counts={'valence':len(datasets['train']),'arousal':len(datasets['train']),'dominance':len(datasets['train']) if normalizer.trained['dominance'] else 0}
checkpoint=save_parallel_checkpoint(model,run_dir/'model.pt',emotion_labels,counts,'trained' if normalizer.trained['dominance'] else 'untrained',column_config=asdict(COLUMNS),vad_normalization=normalizer.to_dict(),encoder_info={'id':encoder_id,'demo_mode':DEMO_MODE,'frozen':True},training_history=history,evaluation_metrics=test_metrics,metadata={'demo_results_not_for_research':DEMO_MODE})
save_json(emotion_labels,run_dir/'emotion_labels.json'); print(run_dir/'model.pt')

## 11. 推論用WAVフォルダの一括処理感情名・全確率・正規化V/A/D・元尺度V/A/D・各状態を表とCSVへ保存します。

In [ ]:
inference_dir=Path(CONFIG.inference_dir); inference_dir=inference_dir if inference_dir.is_absolute() else ROOT/inference_dir
results=predict_wav_folder(inference_dir,model,emotion_labels,normalizer,cache,run_dir/'inference_results.csv',CONFIG.device)
display(results)
if DEMO_MODE: print('注意: デモ結果は研究結果ではありません。')